# IDM-VTON — Gradio App (Public Link)

CatVTON kabi ishlatish: cell 1→5 run qiling, public URL oling, brauzerda test qiling.

**Talablar:** T4 GPU | Google Drive (model caching)

In [ ]:
# Cell 1 — GPU + Drive
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import torch
print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")

In [ ]:
# Cell 2 — Clone
%cd /content
!rm -rf /content/IDM-VTON
!git clone https://github.com/yisol/IDM-VTON.git /content/IDM-VTON
%cd /content/IDM-VTON
print("Clone tayyor.")

In [ ]:
# Cell 3 — Dependencies
!pip install -q omegaconf einops
!pip install -q transformers==4.46.3 accelerate diffusers==0.25.0
!pip install -q torchvision opencv-python Pillow huggingface_hub
!pip install -q onnxruntime-gpu
!pip install -q gradio==3.41.2 gradio-client
!pip install -q basicsr
!pip install -q detectron2 -f https://dl.fbaipublicfiles.com/detectron2/wheels/cu121/torch2.5/index.html

import torch
print(f"torch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")

In [ ]:
# Cell 4 — Modellarni Drive'ga yuklab, ckpt/ ga sozlash
import os, shutil
from huggingface_hub import snapshot_download, hf_hub_download

CKP = "/content/IDM-VTON/ckpt"
os.makedirs(CKP, exist_ok=True)

def drive_symlink(drive_path, local_path):
    os.makedirs(os.path.dirname(local_path), exist_ok=True)
    if os.path.islink(local_path): os.unlink(local_path)
    if os.path.exists(local_path) and not os.path.islink(local_path): shutil.rmtree(local_path)
    os.symlink(drive_path, local_path)

# 1. IDM-VTON asosiy model
DRIVE_IDM = "/content/drive/MyDrive/Lookzi/hf_models/idm-vton"
if not os.path.exists(f"{DRIVE_IDM}/unet"):
    print("IDM-VTON yuklanmoqda (~8GB, 20-30 daqiqa)...")
    snapshot_download("yisol/IDM-VTON", local_dir=DRIVE_IDM)
else:
    print("IDM-VTON: Drive'da mavjud")
drive_symlink(DRIVE_IDM, f"{CKP}/IDM-VTON")

# 2. Humanparsing ONNX
DRIVE_HP = "/content/drive/MyDrive/Lookzi/hf_models/humanparsing"
os.makedirs(DRIVE_HP, exist_ok=True)
for fname in ["parsing_atr.onnx", "parsing_lip.onnx"]:
    fpath = f"{DRIVE_HP}/{fname}"
    if not os.path.exists(fpath):
        print(f"{fname} yuklanmoqda...")
        try:
            snapshot_download("levihsu/GarmentDiffusion", local_dir=DRIVE_HP, allow_patterns=[fname])
        except:
            # IDM-VTON repo'da ham bo'lishi mumkin
            alt = f"{DRIVE_IDM}/{fname}"
            if os.path.exists(alt): shutil.copy(alt, fpath)
            else: print(f"  TOPILMADI: {fname} — menga xabar bering")
    else:
        print(f"{fname}: Drive'da mavjud")
drive_symlink(DRIVE_HP, f"{CKP}/humanparsing")

# 3. OpenPose
DRIVE_OP = "/content/drive/MyDrive/Lookzi/hf_models/openpose"
os.makedirs(DRIVE_OP, exist_ok=True)
op_pth = f"{DRIVE_OP}/body_pose_model.pth"
if not os.path.exists(op_pth):
    print("OpenPose yuklanmoqda...")
    try:
        p = hf_hub_download("lllyasviel/ControlNet",
                            "annotator/ckpts/body_pose_model.pth",
                            local_dir="/tmp/op")
        shutil.copy(p, op_pth)
    except Exception as e:
        print(f"  XATO: {e}")
else:
    print("OpenPose: Drive'da mavjud")
os.makedirs(f"{CKP}/openpose/ckpts", exist_ok=True)
op_dst = f"{CKP}/openpose/ckpts/body_pose_model.pth"
if os.path.exists(op_pth) and not os.path.exists(op_dst):
    os.symlink(op_pth, op_dst)

# 4. Holat
print("\n--- Model holati ---")
for name, path in [
    ("IDM-VTON unet",    f"{DRIVE_IDM}/unet"),
    ("parsing_atr.onnx", f"{DRIVE_HP}/parsing_atr.onnx"),
    ("parsing_lip.onnx", f"{DRIVE_HP}/parsing_lip.onnx"),
    ("body_pose_model",  op_pth),
]:
    print(f"  {'OK' if os.path.exists(path) else 'YOQ'}: {name}")

In [ ]:
# Cell 5 — IDM-VTON Gradio App ishga tushirish
# Public link 'Running on public URL: https://...' qatorida chiqadi.
# Uni brauzerda oching va CatVTON kabi test qiling.

!fuser -k 7860/tcp 2>/dev/null; sleep 1

import sys
sys.path.insert(0, '/content/IDM-VTON')

# IDM-VTON gradio demo'sini share=True bilan ishga tushirish
import runpy, gradio as gr

# gradio_demo/app.py ni import qilamiz, launch() ni o'zimiz chaqiramiz
try:
    import gradio_demo.app as idm_app
    # Demo ob'ektini topamiz
    demo = getattr(idm_app, 'demo', None)
    if demo is None:
        raise AttributeError("'demo' topilmadi — app.py tuzilishini tekshiring")
    print("\nGradio app ishga tushmoqda...")
    demo.queue().launch(
        server_name="0.0.0.0",
        server_port=7860,
        share=True,
        show_error=True,
    )
except Exception as e:
    import traceback
    print(f"XATO: {e}")
    traceback.print_exc()
    print("\n--- Xato matnini menga yuboring ---")